# Word Embeddings

## Definition
Word embeddings are dense, low-dimensional vector representations of words in a continuous vector space, where semantically similar words are mapped to nearby points. Unlike one-hot encoding, embeddings capture semantic and syntactic relationships between words. They are learned from large text corpora using neural networks or matrix factorization techniques.

## Why It Is Needed
- **Semantic Representation:** Captures meaning — `king - man + woman ≈ queen` — which sparse representations like one-hot or Bag-of-Words cannot express.
- **Dimensionality Reduction:** Reduces vocabulary-sized one-hot vectors (e.g., 50,000 dimensions) to compact dense vectors (e.g., 100–300 dimensions).
- **Transfer Learning:** Pre-trained embeddings encode knowledge from massive corpora and can be reused across tasks, reducing the need for large labeled datasets.

## Real-World Applications
- Semantic search and document retrieval systems
- Text classification and sentiment analysis models
- Machine translation input representations
- Recommendation systems using item and user text descriptions
- Question answering and reading comprehension models

## Important Points
- **One-Hot Encoding vs. Embeddings:**

| Feature | One-Hot | Word Embedding |
|---|---|---|
| Dimensionality | Vocabulary size | Fixed (e.g., 100–300) |
| Captures Semantics | No | Yes |
| Sparsity | Sparse | Dense |

- **Static Embeddings:** One fixed vector per word regardless of context — Word2Vec, GloVe, FastText.
- **Contextual Embeddings:** Different vectors for the same word in different contexts — ELMo, BERT.
- **Embedding Dimension:** Typically 50, 100, 200, or 300 for static; 768 or 1024 for transformer-based.
- **Cosine Similarity:** Measures the similarity between two word vectors:
  - `similarity = cos(θ) = (A · B) / (||A|| × ||B||)`

## Visual Understanding
```
[Insert 2D t-SNE visualization showing word clusters in embedding space —
 e.g., countries clustered together, fruits clustered together,
 demonstrating that semantically related words are geometrically close]
```

## Implementation
Practical implementation will be added here.

## Key Takeaways
- Word embeddings represent words as dense, continuous vectors capturing semantic meaning.
- The famous analogy `king - man + woman ≈ queen` demonstrates the power of embeddings.
- Static embeddings (Word2Vec, GloVe) assign one vector per word; contextual embeddings (BERT) vary by context.
- Cosine similarity is the standard metric for comparing word vectors.
- Pre-trained embeddings provide a powerful starting point for any NLP task.

In [1]:
!pip install gensim


  Using cached gensim-4.4.0-cp310-cp310-win_amd64.whl.metadata (8.6 kB)
Using cached gensim-4.4.0-cp310-cp310-win_amd64.whl (24.4 MB)


In [5]:
import numpy as np
import gensim.downloader as api 

import numpy as np
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score, recall_score, f1_score

# Load word embeddings (can replace with your own Word2Vec/GloVe)
model = api.load('glove-wiki-gigaword-100')  # or 'word2vec-google-news-300'



In [6]:
import gensim.downloader as api

print(api.info()['models']['glove-wiki-gigaword-100'])

{'num_records': 400000, 'file_size': 134300434, 'base_dataset': 'Wikipedia 2014 + Gigaword 5 (6B tokens, uncased)', 'reader_code': 'https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/__init__.py', 'license': 'http://opendatacommons.org/licenses/pddl/', 'parameters': {'dimension': 100}, 'description': 'Pre-trained vectors based on Wikipedia 2014 + Gigaword 5.6B tokens, 400K vocab, uncased (https://nlp.stanford.edu/projects/glove/).', 'preprocessing': 'Converted to w2v format with `python -m gensim.scripts.glove2word2vec -i <fname> -o glove-wiki-gigaword-100.txt`.', 'read_more': ['https://nlp.stanford.edu/projects/glove/', 'https://nlp.stanford.edu/pubs/glove.pdf'], 'checksum': '40ec481866001177b8cd4cb0df92924f', 'file_name': 'glove-wiki-gigaword-100.gz', 'parts': 1}


In [7]:
def get_sentence_vector(sentence):
    words = [word for word in sentence.lower().split() if word in model]
    if not words:
        return np.zeros(model.vector_size)
    return np.mean([model[word] for word in words], axis=0)

## Segmenting similar tasks

In [8]:
tasks = [
    "clean data", "build model", "train algorithm", "remove outliers",
    "deploy model", "visualize data", "tune hyperparameters", "generate report"
]

task_vectors = [get_sentence_vector(task) for task in tasks]

# Use KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(task_vectors)

# Print grouped tasks
for i, label in enumerate(labels):
    print(f"Cluster {label}: {tasks[i]}")

Cluster 1: clean data
Cluster 0: build model
Cluster 0: train algorithm
Cluster 1: remove outliers
Cluster 0: deploy model
Cluster 1: visualize data
Cluster 2: tune hyperparameters
Cluster 1: generate report


# Checking accuracy in summary

In [9]:
def check_summary_accuracy(original_text, summary_text):
    orig_words = [word for word in original_text.lower().split() if word in model]
    summary_words = [word for word in summary_text.lower().split() if word in model]

    orig_vecs = np.array([model[word] for word in orig_words])
    summary_vecs = np.array([model[word] for word in summary_words])

    similarities = cosine_similarity(summary_vecs, orig_vecs)
    coverage = np.mean(np.max(similarities, axis=1))  # max sim for each summary word
    return coverage

original = "The quick brown fox jumps over the lazy dog and runs into the forest."
summary = "A brown fox jumps over a lazy dog."

accuracy_score = check_summary_accuracy(original, summary)
print("Summary accuracy score (cosine-based):", accuracy_score)

Summary accuracy score (cosine-based): 0.93599373


# Text Clustering

In [10]:
documents = [
    "Dogs are wonderful pets",
    "Cats are independent animals",
    "Dogs love to play fetch",
    "Cats love to nap",
    "Football is a great sport",
    "Soccer is popular worldwide",
]

doc_vectors = [get_sentence_vector(doc) for doc in documents]
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(doc_vectors)

for i, label in enumerate(clusters):
    print(f"Cluster {label}: {documents[i]}")


Cluster 0: Dogs are wonderful pets
Cluster 0: Cats are independent animals
Cluster 0: Dogs love to play fetch
Cluster 0: Cats love to nap
Cluster 1: Football is a great sport
Cluster 1: Soccer is popular worldwide


# Semantic Search Result

In [11]:
corpus = [
    "How to train a neural network",
    "Ways to clean data for machine learning",
    "Data visualization techniques",
    "Best practices for model deployment",
]

query = "visualizing data"
query_vec = get_sentence_vector(query)

corpus_vecs = [get_sentence_vector(doc) for doc in corpus]
sims = cosine_similarity([query_vec], corpus_vecs)[0]

results = sorted(zip(corpus, sims), key=lambda x: x[1], reverse=True)
print("Semantic Search Results:")
for doc, score in results:
    print(f"{score:.3f}: {doc}")

Semantic Search Results:
0.829: Data visualization techniques
0.634: Ways to clean data for machine learning
0.515: How to train a neural network
0.438: Best practices for model deployment
